In [ ]:
import glob
import ast
import time
import numpy as np
import pandas as pd
from tqdm import tqdm 
import cv2
from PIL import Image
import re
import joblib
import matplotlib.pyplot as plt
import gc
import zipfile

from tqdm import tqdm
import shutil

In [ ]:
import warnings
warnings.filterwarnings('ignore')

### Loading package

In [ ]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[2]
sys.path.append(str(repo_path))

In [ ]:
from py.utils import verifyDir,verifyFile

In [ ]:
from py.config import Config

cfg = Config()

np.random.seed(cfg.RANDOM_STATE)
cfg.DATA_PATH, cfg.MODEL_PATH

In [ ]:
QSCORE_PATH=f"{cfg.DATA_PATH}pp2/Qscores/"
IMAGES_PATH = f"{cfg.DATA_PATH}pp2/images/"

ADE20K_DIR = f"{cfg.DATA_PATH}{cfg.SEG_DATASET}/"
SEGMENT_DIR = f"{cfg.DATA_PATH}pp2/segmentations/{cfg.SEG_DATASET}/{cfg.SEG_MODEL_NAME}/"

UPD4K_DIR = f"{cfg.DATA_PATH}/{cfg.UPD_DATASET}/"
UPD_SEGMENT_DIR = f"{cfg.DATA_PATH}UrbanPhysicalDisorder/{cfg.UPD_DATASET}/"

In [ ]:
verifyDir(UPD_SEGMENT_DIR)

### Loading data

In [ ]:
from py.datasets import UrbanPhysicalDisorder

uss = UrbanPhysicalDisorder(data_path=cfg.DATA_PATH)
uss.generate_dataset(dataset=f"{cfg.SEG_DATASET}_{cfg.UPD_DATASET}")

objects_df = uss.get_urban_street_categories()
objects_df

### Merging masks

In [ ]:
import fnmatch

In [ ]:
columns_to_keep = ["image_id", "seg_image_path", "seg_overlay_image_path", "mask_path", "ratio_path"]

In [ ]:
%%time
segment_df = pd.DataFrame()

for idx, current_city in enumerate(["Rio De Janeiro"]):
    print(f"{idx+1}: Evaluating {current_city}...")
    
    OUT_DIR = f"{UPD_SEGMENT_DIR}{current_city}/"
    
    verifyDir(OUT_DIR)
    verifyDir(f"{OUT_DIR}/masks/")
    verifyDir(f"{OUT_DIR}/segmented_images/")
    verifyDir(f"{OUT_DIR}/segmented_images_overlay/")
    verifyDir(f"{OUT_DIR}/ratios/")

    city_segment_df = pd.DataFrame()
    
    with zipfile.ZipFile(f'{UPD4K_DIR}/{current_city}.zip', 'r') as zip_ref:
        # Get all PNG files in SegmentationClass directory
        manual_img_seg_list = np.sort([f'{UPD4K_DIR}{f}' for f in zip_ref.namelist() 
                     if fnmatch.fnmatch(f, f'{current_city}/SegmentationClass/*.png')])
    
    manual_img_seg_ids = [ img.split("/")[-1].replace(".png", "") for img in manual_img_seg_list]
    
    img_seg_list = np.sort([f for f in glob.glob(f'{SEGMENT_DIR}/{current_city}/segmented_images/*.png') ])
    img_seg_ids = [ img.split("/")[-1].replace(".png", "") for img in img_seg_list]
    
    for img_path in tqdm(img_seg_list):
        
        current_id = img_path.split("/")[-1].replace(".JPEG", "").replace(".png", "")
        current_image = Image.open(glob.glob(f'{IMAGES_PATH}/{current_city}/{current_id}.JPG')[0]).convert("RGB")

        if verifyFile(f"{OUT_DIR}/masks/{current_id}.pkl") and verifyFile(f"{OUT_DIR}/ratios/{current_id}.csv") and verifyFile(f"{OUT_DIR}/segmented_images/{current_id}.png") and verifyFile(f"{OUT_DIR}/segmented_images_overlay/{current_id}.png"):
            ratio_df = pd.read_csv(f"{OUT_DIR}/ratios/{current_id}.csv", sep=";", low_memory=False)
            
        else:
            if current_id in manual_img_seg_ids:
                manual_img_seg = Image.open(
                                    zipfile.ZipFile(f'{UPD4K_DIR}/{current_city}.zip')
                                    .open(f'{current_city}/SegmentationClass/{current_id}.png')
                                ).convert("RGB")
                
                colors_tuple = uss.calculate_unique_colors(manual_img_seg, to_tuple=True)
        
                # Calculate manual masks ratios
                manual_masks = uss.convert_mask_to_matrix(manual_img_seg, colors_tuple, objects_df)
                
                # Calculate real masks ratios
                model_masks = joblib.load(glob.glob(f'{SEGMENT_DIR}/{current_city}/masks/{current_id}.pkl')[0])
                
                # merging masks
                merged_masks = np.where(manual_masks != 0, manual_masks, model_masks)
                joblib.dump(merged_masks, f"{OUT_DIR}/masks/{current_id}.pkl")
                ratio_df = uss.calculate_pixel_ratios(merged_masks, objects_df)
                ratio_df.to_csv(f"{OUT_DIR}/ratios/{current_id}.csv", sep=";", index=False)
        
                # merging both images
                ade20k_img_seg = Image.open(glob.glob(f'{SEGMENT_DIR}/{current_city}/segmented_images/{current_id}.png')[0]).convert("RGB")
                merged_img_seg = uss.merge_segmentations(ade20k_img_seg, manual_img_seg)
                merged_image = Image.fromarray(merged_img_seg)
                merged_image.save(f"{OUT_DIR}/segmented_images/{current_id}.png")
        
                # Overlay image
                image_overlay = Image.blend(current_image, merged_image, alpha=0.6)
                image_overlay.save(f"{OUT_DIR}/segmented_images_overlay/{current_id}.png")
        
            else:
                # copying masks
                model_masks = joblib.load(glob.glob(f'{SEGMENT_DIR}/{current_city}/masks/{current_id}.pkl')[0])
                source = glob.glob(f'{SEGMENT_DIR}/{current_city}/masks/{current_id}.pkl')[0]
                destination = f"{OUT_DIR}/masks/{current_id}.pkl"
                shutil.copy(source, destination)
        
                # copying ratios
                source = glob.glob(f'{SEGMENT_DIR}/{current_city}/ratios/{current_id}.csv')[0]
                destination = f"{OUT_DIR}/ratios/{current_id}.csv"
                shutil.copy(source, destination)
        
                # copying image segmentation
                source = glob.glob(f'{SEGMENT_DIR}/{current_city}/segmented_images/{current_id}.png')[0]
                destination = f"{OUT_DIR}/segmented_images/{current_id}.png"
                shutil.copy(source, destination)
        
                # copying image overlay
                source = glob.glob(f'{SEGMENT_DIR}/{current_city}/segmented_images_overlay/{current_id}.png')[0]
                destination = f"{OUT_DIR}/segmented_images_overlay/{current_id}.png"
                shutil.copy(source, destination)
        
                # read ratios to append
                ratio_df = pd.read_csv(f'{SEGMENT_DIR}/{current_city}/ratios/{current_id}.csv', sep=";", low_memory=False)
    
        # merge segmentations
        df_pivot = uss.parse_ratios(current_id, ratio_df)
        df_pivot["seg_image_path"] = f"{current_city}/segmented_images/{current_id}.png"
        df_pivot["seg_overlay_image_path"] = f"{current_city}/segmented_images_overlay/{current_id}.png"
        df_pivot["mask_path"] = f"{current_city}/masks/{current_id}.pkl"
        df_pivot["ratio_path"] = f"{current_city}/ratios/{current_id}.csv"

        city_segment_df = pd.concat([city_segment_df, df_pivot], ignore_index=True)
        city_segment_df = city_segment_df[columns_to_keep + [col for col in city_segment_df.columns if col not in columns_to_keep]].copy()
        city_segment_df.fillna(0, inplace=True)

    city_segment_df.fillna(0, inplace=True)
    city_segment_df.to_csv(f"{OUT_DIR}/segmentations.csv", sep=";", index=False)

    segment_df = pd.concat([segment_df, city_segment_df], ignore_index=True)
    segment_df.fillna(0, inplace=True)

In [ ]:
segment_df

In [ ]:
np.sort(segment_df.columns)

In [ ]:
segment_df.to_csv(f"{UPD_SEGMENT_DIR}/segmentations.csv", sep=";", index=False)

### Summary

In [ ]:
manual_segment_df = segment_df[columns_to_keep + uss.upd4k_labels].copy()

In [ ]:
manual_segment_df.shape, segment_df.shape#, model_segment_df.shape

In [ ]:
set(segment_df.columns) - set(manual_segment_df.columns)#, set(segment_df.columns) - set(model_segment_df.columns)

In [ ]:
metric="safety"

In [ ]:
%%time
data_df = pd.read_csv(f"{QSCORE_PATH}scores.csv", sep=";", low_memory=False)
manual_segmentation_df = pd.merge(data_df, manual_segment_df, on="image_id", how="inner")
manual_segmentation_df.sort_values(by=metric, inplace=True, ascending=False)
manual_segmentation_df

### Object Presence

#### All samples

In [ ]:
manual_segmentation_df.info()

In [ ]:
feature_presence = uss.features_presence(manual_segmentation_df.iloc[:, 17:])
feature_presence[0] = feature_presence[0]*100
uss.print_object_present(feature_presence, fig_size=(16,20))

#### Positive samples

In [ ]:
feature_presence = uss.features_presence(manual_segmentation_df[manual_segmentation_df[metric]>=6].iloc[:, 17:])
uss.print_object_present(feature_presence, fig_size=(16,20))

#### Negative samples

In [ ]:
feature_presence = uss.features_presence(manual_segmentation_df[manual_segmentation_df[metric]<4].iloc[:, 17:])
uss.print_object_present(feature_presence, fig_size=(16,20))

### Boxplots

#### All samples

In [ ]:
uss.print_object_boxplot(manual_segmentation_df.iloc[:, 17:]*100, fig_size=(25, 20))

#### Positive samples

In [ ]:
uss.print_object_boxplot(manual_segmentation_df[manual_segmentation_df[metric]>=6].iloc[:, 17:]*100, fig_size=(25, 20))

#### Negative samples

In [ ]:
uss.print_object_boxplot(manual_segmentation_df[manual_segmentation_df[metric]<4].iloc[:, 17:]*100, fig_size=(25, 20))